# 第二章实验：输出层反向传播

固定窗口模型沿用 `forward.py` 的参数，输入“我 喜欢”，目标为“数学”。先求交叉熵对 logits 的梯度，再传给输出层权重、偏置和隐藏表示。这里只更新输出层，embedding 和第一层保持固定。

[第二章正文](../../docs/02-neural-language-models.md) · [前向计算](forward.ipynb)

## 1. 前向计算与 logits 梯度

单样本交叉熵为 $\mathcal L=-\ln q_y$，梯度为 $g_z=q-r$，其中 $r$ 是目标标签的 one-hot 向量。计算使用 CPU、NumPy float64、batch size 1、窗口长度 2、词表大小 6、embedding 维度 2、隐藏宽度 3；参数固定，无随机采样和数据划分。

In [1]:
from pathlib import Path
import platform
import sys
import numpy as np

root = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents]
            if (candidate / 'docs/00-learning-path.md').exists())
sys.path.insert(0, str(root / 'experiments/02-neural-lm'))
from forward import forward, W2, B2

np.set_printoptions(precision=6, suppress=True)
result = forward([[0, 2]])
hidden = result['hidden']
grad_logits = result['probs'].copy()
grad_logits[0, 4] -= 1
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('dtype:', hidden.dtype)
print('hidden:', hidden)
print('probabilities:', result['probs'])
print('logits gradient:', grad_logits)
print('NLL:', -result['log_probs'][0, 4])

Python: 3.14.2 NumPy: 2.3.5
dtype: float64
hidden: [[0.800499 0.       0.197375]]
probabilities: [[0.065385 0.06945  0.121519 0.134271 0.310868 0.298507]]
logits gradient: [[ 0.065385  0.06945   0.121519  0.134271 -0.689132  0.298507]]
NLL: 1.1683863193730513


## 2. 输出层的三个梯度

前向为 $z=hW_2+b_2$。逐元素应用链式法则：

$$\frac{\partial\mathcal L}{\partial(W_2)_{ij}}=h_i(g_z)_j,\qquad \frac{\partial\mathcal L}{\partial(b_2)_j}=(g_z)_j,\qquad \frac{\partial\mathcal L}{\partial h_i}=\sum_j(g_z)_j(W_2)_{ij}.$$

因此矩阵形式为 $g_{W_2}=h^\top g_z$、$g_{b_2}=g_z$、$g_h=g_zW_2^\top$。代码中偏置是形状 `[6]` 的数组，因此单样本偏置梯度取 `grad_logits[0]`。

In [2]:
grad_weight = hidden.T @ grad_logits
grad_bias = grad_logits[0].copy()
grad_hidden = grad_logits @ W2.T
print('W2 gradient', grad_weight.shape, '\n', grad_weight)
print('b2 gradient', grad_bias.shape, '\n', grad_bias)
print('hidden gradient', grad_hidden.shape, '\n', grad_hidden)

W2 gradient (3, 6) 
 [[ 0.052341  0.055595  0.097276  0.107483 -0.551649  0.238955]
 [ 0.        0.        0.        0.        0.        0.      ]
 [ 0.012905  0.013708  0.023985  0.026502 -0.136018  0.058918]]
b2 gradient (6,) 
 [ 0.065385  0.06945   0.121519  0.134271 -0.689132  0.298507]
hidden gradient (1, 3) 
 [[-0.363817 -0.129406 -0.117033]]


## 3. 用中心差分独立核对

对每个变量分别扰动 $\pm\varepsilon$，保持其他变量固定，用 $[\mathcal L(v+\varepsilon)-\mathcal L(v-\varepsilon)]/(2\varepsilon)$ 近似偏导。这里取 $\varepsilon=10^{-5}$，检查全部 18 个输出层权重、6 个偏置和3 个隐藏坐标。

In [3]:
def output_loss(weight, bias, features):
    logits = (features @ weight + bias)[0]
    shifted = logits - logits.max()
    return np.log(np.exp(shifted).sum()) - shifted[4]


def central_difference(values, evaluate, epsilon=1e-5):
    numerical = np.zeros_like(values)
    for index in np.ndindex(values.shape):
        plus = values.copy()
        minus = values.copy()
        plus[index] += epsilon
        minus[index] -= epsilon
        numerical[index] = (evaluate(plus) - evaluate(minus)) / (2 * epsilon)
    return numerical


checks = [
    ('W2', grad_weight, central_difference(W2, lambda values: output_loss(values, B2, hidden))),
    ('b2', grad_bias, central_difference(B2, lambda values: output_loss(W2, values, hidden))),
    ('hidden', grad_hidden, central_difference(hidden, lambda values: output_loss(W2, B2, values))),
]
for name, analytical, numerical in checks:
    np.testing.assert_allclose(analytical, numerical, rtol=0, atol=1e-9)
    print(name, 'max absolute error:', np.max(np.abs(analytical - numerical)))

W2 max absolute error: 8.331273171346965e-12
b2 max absolute error: 5.721464968466705e-12
hidden max absolute error: 8.75874373029717e-12


## 4. 只更新输出层，再做前向计算

使用学习率 $\eta=0.1$，同时更新 $W_2,b_2$。两个梯度及 $g_h$ 均由更新前的同一次前向计算产生。此处不更新隐藏向量，它仍由固定的 embedding 和第一层参数计算得到；$g_h$ 用于后续向更早的层传播。

In [4]:
learning_rate = 0.1
updated_weight = W2 - learning_rate * grad_weight
updated_bias = B2 - learning_rate * grad_bias
updated_logits = hidden @ updated_weight + updated_bias
shifted = updated_logits - updated_logits.max(axis=-1, keepdims=True)
updated_probs = np.exp(shifted) / np.exp(shifted).sum(axis=-1, keepdims=True)
before = output_loss(W2, B2, hidden)
after = output_loss(updated_weight, updated_bias, hidden)
print('W2[0, math]:', W2[0, 4], '->', updated_weight[0, 4])
print('b2[math]:', B2[4], '->', updated_bias[4])
print('P(math):', result['probs'][0, 4], '->', updated_probs[0, 4])
print('NLL:', before, '->', after)
assert after < before

W2[0, math]: 0.8 -> 0.8551649348818254
b2[math]: 0.1 -> 0.16891318213043516
P(math): 0.31086817869564864 -> 0.34332572006191575
NLL: 1.1683863193730513 -> 1.0690756612493693


## 练习

1. 第二个隐藏坐标为零时，输出层权重第二行的梯度是什么？这是否意味着该隐藏坐标的梯度也为零？分别查看两条链式法则。
2. 如果某个隐藏坐标为负，目标词对应的权重一定要增加才能提高目标分数吗？

下一步将 $g_h$ 继续通过 $h=\tanh(a)$ 传给第一层和 embedding，完成全模型反向传播。